# Data Microstructure

In [ ]:
# General TODOs:

# TODO: ...

# General NOTEs:

# NOTE: ...

# Setup

## Imports

In [ ]:
import warnings

import matplotlib.pyplot as plt
import pandas as pd

from thesis_project import (
    config,
)
from thesis_project.collectors import (
    run_collectors_double,
)

## Paths

In [ ]:
TICKER = "fbtp"
NOTEBOOK = "data-microstructure"

# Lobs directories
LOB_DIR = config.RAW_DIR / TICKER
LOB_FUT_DIR = LOB_DIR / "futures"
LOB_CTD_DIR = LOB_DIR / "ctd"

# Output directories
if NOTEBOOK is not None:
    # OUT_FIG_DIR = config.FIG_DIR / TICKER / NOTEBOOK
    # OUT_FIG_DIR.mkdir(parents=True, exist_ok=True)
    # OUT_TAB_DIR = config.TAB_DIR / TICKER / NOTEBOOK
    # OUT_TAB_DIR.mkdir(parents=True, exist_ok=True)
    OUT_INT_DIR = config.INT_DIR / TICKER / NOTEBOOK
    OUT_INT_DIR.mkdir(parents=True, exist_ok=True)
    # OUT_PRO_DIR = config.PRO_DIR / TICKER / NOTEBOOK
    # OUT_PRO_DIR.mkdir(parents=True, exist_ok=True)

## Settings

In [ ]:
# Matplotlib global settings
# config.MPL contains a personalized default configuration for matplotlib
plt.rcdefaults()
plt.style.use("seaborn-v0_8-paper")

In [ ]:
# Warnings settings
warnings.simplefilter("always", UserWarning)

In [ ]:
# Sample date for quick analysis
year = "2022"
month = "09"
day = "07"
date_str = f"{year}_{month}_{day}"
fut_filename = f"futures_lob_freq_1s_{date_str}.parquet"
ctd_filename = f"ctd_lob_freq_1s_{date_str}.parquet"

## Utilities

### Plotting

# Main

## Common

In [ ]:
# Common data
cf_df = pd.read_csv(
    LOB_DIR / "daily_cf.csv", parse_dates=["Date"], date_format="%Y-%m-%d", index_col="Date"
)

ctd_meta_df = pd.read_csv(
    LOB_DIR / "ctd_metadata.csv", parse_dates=["Issue Date", "Maturity Date", "First Coupon Date"]
)

fut_meta_df = pd.read_csv(
    LOB_DIR / "fut_metadata.csv", parse_dates=["Last Trading Date", "Delivery Date"]
)

ctd_switch_df = pd.read_csv(LOB_DIR / "ctd_switch.csv", parse_dates=["CTD Switch Date"])

## Gross Basis

In [ ]:
from thesis_project.collectors import compute_gross_basis

context = {
    "CF": cf_df,
}

results = run_collectors_double(
    LOB_FUT_DIR,
    LOB_CTD_DIR,
    extractors={"Basis": compute_gross_basis},
    start_year=config.FBTP_START_YEAR,
    end_year=config.FBTP_END_YEAR,
    start_time=config.CTD_MRKT_OPEN,
    end_time=config.CTD_MRKT_CLOSE,
    context=context,
)

In [ ]:
data = results["Basis"]

start_date = data.index[0].strftime("%Y-%m-%d").replace("-", "_")
end_date = data.index[-1].strftime("%Y-%m-%d").replace("-", "_")

name = "basis"
freq = "1s"
filename = config.SERIES_FILENAME_TEMPLATE.format(
    series_name=name, freq=freq, start_date=start_date, end_date=end_date, ext="parquet"
)

subdir = "basis"
# data.to_parquet(OUT_INT_DIR / subdir / filename)

## Implied Repo Rate

In [ ]:
from thesis_project.collectors import compute_implied_repo

context = {
    "CF": cf_df,
    "CTD Metadata": ctd_meta_df,
    "FUT Metadata": fut_meta_df,
    "Frequency": "1s",
    "Notional": 100.0,
}

results = run_collectors_double(
    LOB_FUT_DIR,
    LOB_CTD_DIR,
    extractors={"IRR": compute_implied_repo},
    start_year=config.FBTP_START_YEAR,
    end_year=config.FBTP_END_YEAR,
    start_time=config.CTD_MRKT_OPEN,
    end_time=config.CTD_MRKT_CLOSE,
    context=context,
)

In [ ]:
data = results["IRR"]

start_date = data.index[0].strftime("%Y-%m-%d").replace("-", "_")
end_date = data.index[-1].strftime("%Y-%m-%d").replace("-", "_")

name = "irr"
freq = context["Frequency"]
filename = config.SERIES_FILENAME_TEMPLATE.format(
    series_name=name, freq=freq, start_date=start_date, end_date=end_date, ext="parquet"
)

subdir = "irr"
data.to_parquet(OUT_INT_DIR / subdir / filename)